---
title: "Part 9: Pandas Operations"
---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sambaiga/ds-mlops-path/blob/main/tutorials/01-python-basics/09-pandas-operations.ipynb) [![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue.svg?logo=jupyter&logoColor=white)](https://raw.githubusercontent.com/sambaiga/ds-mlops-path/main/tutorials/01-python-basics/09-pandas-operations.ipynb)

**DS-MLOps Data Analysis**

**Python 3.12+ | Author: Anthony Faustine**

The CSV is loaded and clean. Now the questions get specific: which students went from failing to passing between semesters? Which course names have a trailing space or a typo? What is the 10-student rolling average for each program? None of these answers live in a raw column -- you have to derive them.

Part 8 covered loading, selecting, filtering, and fixing missing values: the shape and structure of a DataFrame. This notebook covers the operations that turn raw columns into derived ones: applying your own functions row-by-row, counting categorical values, cleaning text, summarizing, and computing window statistics. Part 10 (`10-combining-reshaping.ipynb`) builds on this with `groupby`, merge, pivot, and time-series reshaping.

> Callout markers: [book cover page](../../index.qmd#callout-guide).

::: {.callout-note collapse="true" icon=false}
## Before You Begin

This notebook follows Part 8. It reloads the dataset from scratch, so you can run it independently. Install pandas 3 with `pip install 'pandas>=3'`.
:::

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv("data/university_analytics.csv")
df["average_marks"] = (df["midterm_score"] + df["final_score"] + df["project_score"]) / 3
df.head()

## What's New in Pandas 3

This series is written against pandas 3, released in 2025, and two of its changes are worth understanding before going further, because they change what you see in everyday output.

Run `df.dtypes` and look at the text columns. In pandas 2 they showed up as `object`, the catch-all dtype for anything that isn't a fixed-width number. Pandas 3 gives strings a dedicated `str` dtype instead, backed by PyArrow when it's installed:

In [ ]:
df.dtypes

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> Key Concept: A dedicated <code>str</code> dtype, not <code>object</code></span><br><br>
<code>object</code> dtype could hold strings, but it could just as easily hold a mix of strings, numbers, and Python objects, with no guarantee which. The new <code>str</code> dtype can only hold strings or missing values, so a column typed <code>str</code> is a stronger guarantee than <code>object</code> ever was, and the PyArrow backing makes string operations faster. This is the same dtype the <code>.str</code> accessor in Sec. 3 operates on.
</div>

The second change doesn't show up in any output, it changes how assignment behaves. Pandas 3 makes Copy-on-Write the only behaviour: selecting a subset of a DataFrame never lets you accidentally modify the original through it.

In [ ]:
subset = df[["student_id", "average_marks"]]
subset.loc[0, "average_marks"] = 0.0
print(f"original unaffected: {df.loc[0, 'average_marks']}")
print(f"subset changed     : {subset.loc[0, 'average_marks']}")

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> Key Concept: Copy-on-Write removes the <code>SettingWithCopyWarning</code> guessing game</span><br><br>
Before pandas 3, whether <code>subset</code> above was a view into <code>df</code> or an independent copy depended on details of how <code>subset</code> was created, the source of the infamous <code>SettingWithCopyWarning</code>. Pandas 3 makes every such selection behave as an independent copy: modifying <code>subset</code> never touches <code>df</code>. The warning is gone because the ambiguity it warned about is gone.
</div>

<div style='background:#F5F3FF;border-left:5px solid #7C3AED;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#5B21B6;font-weight:bold'><i class="bi bi-lightbulb-fill"></i> Pro Tip: <code>pd.col()</code> previews the expression style Polars uses</span><br><br>
Pandas 3 adds <code>pd.col("name")</code> as an alternative to a <code>lambda</code> inside <code>.assign()</code>: <code>df.assign(c=pd.col("a") + pd.col("b"))</code> instead of <code>df.assign(c=lambda d: d["a"] + d["b"])</code>. It reads closer to the column-expression style used throughout Polars, covered in Part 11 of this series, and is worth recognising even though this notebook keeps using <code>apply</code> and <code>lambda</code>, the more widely used style today.
</div>

::: {.callout-note collapse="true" icon=false}
## Learning Objectives

By the end of Part 9 you will be able to:

| # | Skill | Covered in |
|---|---|---|
| 1 | Recognise pandas 3's `str` dtype and Copy-on-Write behaviour | Sec. 0 |
| 2 | Apply your own function to a Series or DataFrame with `map` and `apply` | Sec. 1 |
| 3 | Count, filter, and encode categorical columns | Sec. 2 |
| 4 | Clean and query text columns with the `.str` accessor | Sec. 3 |
| 5 | Summarize a dataset with descriptive statistics in one call | Sec. 4 |
| 6 | Compute rolling and expanding window statistics | Sec. 5a |
| 7 | Write readable multi-condition filters with `df.query()` | Sec. 5b |
:::


## 1. Function Application with `map` and `apply`

NumPy's vectorized operations cover arithmetic and comparisons, but sometimes the transformation you need is your own logic: a lookup, a multi-branch rule, a calculation that combines several columns per row. `map` and `apply` are how pandas runs that logic.

`Series.map()` is for simple one-to-one substitution: pass it a dict (or a function) and it replaces every value with its mapped equivalent. Recoding the `gender` column's short codes into full labels is a `map` problem, not an `apply` one:

In [ ]:
gender_labels = {"M": "Male", "F": "Female"}
df["gender_label"] = df["gender"].map(gender_labels)
df[["gender", "gender_label"]].head(3)

`Series.map()` transforms one column. For element-wise operations across ALL columns simultaneously, pandas 3.0 introduced `DataFrame.map()`: the replacement for the deprecated `applymap()`. Same signature, same semantics, new name.

In [ ]:
# Round every numeric cell to 1 decimal place
numeric_cols = df.select_dtypes("number").columns.tolist()
rounded = df[numeric_cols].map(lambda x: round(x, 1) if pd.notna(x) else x)
rounded.head(3)

<div style='background:#FEF2F2;border-left:5px solid #DC2626;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#991B1B;font-weight:bold'><i class="bi bi-bug-fill"></i> Common Mistake: Using <code>applymap()</code> from an older tutorial</span><br><br>
<code>df.applymap()</code> was removed in pandas 3.0. Any code or tutorial older than 2024 that uses <code>applymap</code> will raise <code>AttributeError</code>. The fix is a one-word rename: <code>df.applymap(fn)</code> becomes <code>df.map(fn)</code>.
</div>

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> Key Concept: <code>map</code> is for substitution, <code>apply</code> is for logic</span><br><br>
<code>map</code> expects a dict or a simple function and only ever looks at one value at a time. <code>apply</code> accepts any function, including one with branching logic, and can run over a Series (one value at a time) or a DataFrame (one row or one column at a time, depending on <code>axis</code>). Reach for <code>map</code> first; reach for <code>apply</code> when the rule doesn't fit in a lookup table.
</div>

Converting a normalized mark (0 to 1) into a letter grade needs a multi-branch rule, a job for `apply` with a plain Python function:

In [ ]:
def letter_grade(mark: float) -> str:
    if mark >= 80:
        return "A"
    elif mark >= 60:
        return "B"
    elif mark >= 40:
        return "C"
    else:
        return "D"


df["grade"] = df["average_marks"].apply(letter_grade)
df[["student_id", "average_marks", "grade"]].head()

`np.where` handles one condition. For multiple ordered conditions with different outcomes, `Series.case_when()` (pandas 2.2+) reads like a SQL CASE statement: check cases top-to-bottom, return the value for the first match, and fall back to a default.

In [ ]:
df["performance"] = df["midterm_score"].case_when(
    caselist=[
        (df["midterm_score"] >= 85, "High"),
        (df["midterm_score"] >= 70, "Medium"),
        (df["midterm_score"] >= 55, "Low"),
        (df["midterm_score"] >= 0, "At Risk"),  # catch-all default
    ],
)
df[["midterm_score", "performance"]].head(8)

<div style='background:#F5F3FF;border-left:5px solid #7C3AED;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#5B21B6;font-weight:bold'><i class="bi bi-lightbulb-fill"></i> Pro Tip: <code>case_when()</code> for three or more branches instead of chained <code>np.where()</code></span><br><br>
<code>case_when()</code> evaluates conditions in order and stops at the first match, like <code>if/elif/else</code>. The <code>default</code> argument fills any row where no condition matched. Use it instead of chained <code>np.where()</code> calls whenever you have three or more branches.
</div>

`DataFrame.apply(..., axis=1)` runs a function once per row, with the whole row available as a Series. Use it when the rule needs more than one column at a time, for example flagging students who are both low-scoring and without internet access:

In [ ]:
def at_risk(row: pd.Series) -> bool:
    return row["average_marks"] < 40 and not row["has_internet"]


df["at_risk"] = df.apply(at_risk, axis=1)
df["at_risk"].sum()

<div style='background:#FEF2F2;border-left:5px solid #DC2626;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#991B1B;font-weight:bold'><i class="bi bi-bug-fill"></i> Common Mistake: Reaching for <code>apply(axis=1)</code> when a vectorized operation already does the job</span><br><br>
<code>df.apply(at_risk, axis=1)</code> calls a Python function once per row, 17,190 times here, which is far slower than an equivalent boolean mask: <code>(df["average_marks"] &lt; 0.4) &amp; (df["has_internet"] == 0)</code> computes the same result with NumPy operating on whole columns at once. Use <code>apply</code> when the rule can't be written with column-wise operations and comparisons; reach for masking first.
</div>

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 1 - Grade Distribution</span><br><br>

<b>Goal:</b> Write a function that returns <code>True</code> for marks of 0.6 or higher and <code>False</code> otherwise, apply it to <code>average_marks</code> to create a new column <code>passed</code>, then print how many students passed.
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>def passed_threshold(mark: float) -> bool:
    ...


df["passed"] = df["average_marks"].apply(passed_threshold)
df["passed"].sum()
# -> 6011</pre>
</div>

In [ ]:
# TODO: write passed_threshold, apply it, and print the count of students who passed
...

## 2. Unique Values, Value Counts, and Membership

`.unique()`, `.value_counts()`, and `.isin()` are usually the first three calls worth making on any categorical column, before any filtering or grouping.

In [ ]:
df["program"].unique()

`.value_counts()` counts how many rows fall into each category, sorted from most to least common. Passing `normalize=True` turns the counts into proportions, which is what you want for a question like "what fraction of students dropped out?":

<div style='background:#EAF7F0;border-left:5px solid #059669;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#059669;font-weight:bold'><i class="bi bi-journal-code"></i> Example: Pass rate from <code>value_counts</code></span><br><br>
<code>df["passed"].value_counts(normalize=True)</code> answers the pass-rate question in a single line, no manual division required.
</div>

In [ ]:
df["passed"].value_counts(normalize=True)

`.isin()` filters rows whose value is in a given list, the categorical equivalent of a boolean comparison. Filtering to the two largest programs combines `.value_counts()` and `.isin()` directly:

In [ ]:
top_programs = df["program"].value_counts().head(2).index
df_top_programs = df[df["program"].isin(top_programs)]
df_top_programs["program"].value_counts()

<div style='background:#F5F3FF;border-left:5px solid #7C3AED;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#5B21B6;font-weight:bold'><i class="bi bi-lightbulb-fill"></i> Pro Tip: <code>.nunique()</code> before <code>.unique()</code> on a column you have not seen yet</span><br><br>
<code>.nunique()</code> returns just the count of distinct values. Running it before <code>.unique()</code> tells you whether printing every unique value is even a reasonable idea, useful for a column that turns out to have 4 categories versus one that turns out to have 4,000.
</div>

A column with a small, fixed set of values is a candidate for the `category` dtype: pandas stores each value once and keeps a compact integer code per row instead of repeating the full string, which is most of `gender`, `program`, and `guardian` here:

In [ ]:
df["program"] = df["program"].astype("category")
df["program"].dtype

Most ML models need numbers, not category labels. `pd.get_dummies()` one-hot encodes a categorical column into one binary column per category, the standard first step before fitting a model on top of this data:

In [ ]:
program_dummies = pd.get_dummies(df["program"], prefix="program")
program_dummies.head()

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> Key Concept: <code>category</code> dtype for memory, one-hot encoding for models</span><br><br>
<code>.astype("category")</code> is about storage and speed: it doesn't change what a column means, only how compactly pandas stores it. <code>pd.get_dummies()</code> is about preparing data for a model that expects numeric input: it turns one categorical column into several binary columns. The two are often used together, category dtype while exploring, dummy columns right before training.
</div>

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 2 - Guardian Breakdown</span><br><br>

<b>Goal:</b> Print the value counts for the <code>guardian</code> column, then filter the DataFrame to students whose guardian is <code>"mother"</code> or <code>"father"</code> using <code>.isin()</code>, and print how many rows remain.
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>df["guardian"].value_counts()

parents = df[df["guardian"].isin(["mother", "father"])]
len(parents)</pre>
</div>

In [ ]:
# TODO: print value counts for guardian, then filter to mother/father and print row count
...

## 3. Working with Text Columns: the `.str` Accessor

`student_id` is a string column, and pandas keeps every string method behind a `.str` accessor rather than directly on the Series. The accessor exists because a Series can hold any dtype, `.str` is what tells pandas you specifically want the string-handling behaviour.

<div style='background:#EFF6FF;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-key-fill"></i> Key Concept: .str is a namespace -- call string methods on the whole column at once</span><br><br>
<code>df['col'].str.upper()</code> applies <code>upper()</code> to every element without a Python loop. The <code>.str</code> prefix is required -- <code>df['col'].upper()</code> raises an <code>AttributeError</code> because <code>upper</code> isn't a Series method. Common methods: <code>.str.strip()</code> for whitespace, <code>.str.contains(pattern)</code> for search, <code>.str.extract(r'(pattern)')</code> for regex capture groups.
</div>

<div style='background:#FEF2F2;border-left:5px solid #DC2626;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#991B1B;font-weight:bold'><i class="bi bi-bug-fill"></i> Common Mistake: Calling a string method directly on a Series</span><br><br>
<code>df["student_id"].upper()</code> raises an <code>AttributeError</code>: <code>Series</code> has no <code>upper</code> method. The string methods live on <code>df["student_id"].str</code>, not on the Series itself, because the Series itself is a general-purpose container that happens to hold strings here.
</div>

In [ ]:
df["student_id"].str.upper().head(3)

`.str.len()` gives the length of every string at once, useful for spotting malformed values before they cause problems downstream:

In [ ]:
id_lengths = df["student_id"].str.len()
id_lengths.value_counts()

`.str.extract()` pulls a regex capture group out of every value, the cleanest way to turn a structured string column into a usable numeric one. Every `student_id` here is the letter `S` followed by four digits (`S0001`–`S0400`), so extracting just the digits gives a numeric ID:

In [ ]:
df["student_id"].str.match(r"^S\d{4}$").all()

In [ ]:
numeric_id = df["student_id"].str.extract(r"S(\d+)")[0].astype("Int64")
df["numeric_id"] = numeric_id
df[["student_id", "numeric_id"]].head(3)

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 3 - Validate and Filter by ID</span><br><br>

<b>Goal:</b> Use <code>.str.startswith("S")</code> to confirm every <code>student_id</code> starts with the letter <code>"S"</code>, then use <code>.str.contains()</code> to count how many contain the digit <code>"0"</code> anywhere in the ID.
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>df["student_id"].str.startswith("S").all()
df["student_id"].str.contains("0").sum()</pre>
</div>

In [ ]:
# TODO: confirm every student_id starts with "s", then count IDs containing "0"
...

## 4. Descriptive Statistics and Summarization

`.describe()` from Part 1 summarizes every numeric column with one call. `.agg()` goes a step further: it computes a chosen list of statistics for a chosen set of columns, in whatever combination you ask for.

<div style='background:#EFF6FF;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-key-fill"></i> Key Concept: .describe() is a first look, not a conclusion</span><br><br>
<code>.describe()</code> shows count, mean, std, min, 25th/50th/75th percentile, and max. It skips non-numeric columns by default. <code>.agg(['mean','std','min','max'])</code> gives you the same numbers for only the columns and statistics you care about, in a table you can compare at a glance. <code>.corr()</code> is Pearson correlation by default -- it measures linear relationship, not causation.
</div>

In [ ]:
df[["midterm_score", "final_score", "project_score"]].describe()

`.agg()` accepts a list of function names and applies every one of them to every selected column, returning a single small table instead of one Series per statistic:

In [ ]:
df[["midterm_score", "final_score", "project_score"]].agg(["mean", "std", "min", "max"])

`.corr()` computes the pairwise correlation between numeric columns, a quick way to see whether strong performance in one subject tends to come with strong performance in another:

In [ ]:
df[["midterm_score", "final_score", "project_score"]].corr()

Prefer one `.agg([...])` call over five separate `.mean()`, `.std()`, `.min()` calls. The aggregation table is easier to scan and keeps related numbers in one output.

<div style='background:#F5F3FF;border-left:5px solid #7C3AED;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#5B21B6;font-weight:bold'><i class="bi bi-lightbulb-fill"></i> Pro Tip: Use <code>groupby().transform()</code> to add a group statistic back to the original rows</span><br><br>
<code>groupby().agg()</code> reduces a DataFrame to one row per group. <code>groupby().transform()</code> does the opposite: it computes the same group statistic but returns a Series the exact same length as the original DataFrame, aligned to the original index, so you can assign it as a new column without a merge:

<pre style='background:#F4F5F6;padding:10px;border-radius:4px;font-size:0.9em'># agg → one row per program
df.groupby("program")["average_marks"].agg("mean")  # shape (4,)

# transform → one row per student, aligned to original index
df["caste_mean_marks"] = df.groupby("program")["average_marks"].transform("mean")
# every student's row now has their caste's mean, ready for feature engineering</pre>

Common use: <b>normalising within groups</b>. <code>(df["average_marks"] - df["caste_mean_marks"]) / df.groupby("program")["average_marks"].transform("std")</code> z-scores each student <em>relative to their caste group</em>, not across the whole dataset, in two lines without any merge.
</div>

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 4 - Subject Spread</span><br><br>

<b>Goal:</b> Use <code>.agg()</code> to compute the mean and standard deviation of <code>midterm_score</code>, <code>final_score</code>, and <code>project_score</code> for students with <code>has_internet == True</code> only.
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>with_internet = df[df["has_internet"]]
with_internet[["midterm_score", "final_score", "project_score"]].agg(["mean", "std"])</pre>
</div>

In [ ]:
# TODO: agg mean and std for the three mark columns, restricted to internet == 1
...

## 5. Window Operations and Expressive Queries

Two operations that come up constantly in DS pipelines but don't fit neatly under "string methods" or "statistics": rolling windows for time-aware aggregation, and `df.query()` for readable multi-condition filters.

### 5a. Rolling and Expanding Windows

`rolling(n)` groups each row with the `n-1` rows before it and applies an aggregation. It is the standard tool for smoothing noisy signals, computing moving averages, or creating features that capture recent trend for a time-ordered dataset.

`expanding()` is the cumulative version: each row aggregates everything from the start of the series up to that point. The first row is its own mean; the second is the mean of rows 1-2; and so on.

![Rolling window of size 3 sliding over five days of scores. Window 1 covers days 1-3, window 2 days 2-4, window 3 days 3-5, each producing a mean. The first two results are NaN.](figs/rolling-window.svg){fig-alt="Five day score boxes at top. Three colored brackets slide across: green (days 1-3, mean 80), blue (days 2-4, mean 76.7), purple (days 3-5, mean 75). Amber bar at bottom notes first two results are NaN."}

In [ ]:
# Sort by average_marks to give the rolling window a meaningful order here
df_sorted = df.sort_values("average_marks").reset_index(drop=True)

# 5-row rolling mean: smooth out noise
df_sorted["marks_rolling5"] = df_sorted["average_marks"].rolling(window=5).mean()

# expanding mean: cumulative average as we move through sorted students
df_sorted["marks_cumulative"] = df_sorted["average_marks"].expanding().mean()

df_sorted[["student_id", "average_marks", "marks_rolling5", "marks_cumulative"]].head(8)

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> Key Concept: Rolling windows require ordered data to be meaningful</span><br><br>
<code>rolling(5)</code> looks at the 5 rows immediately before each row in whatever order they sit in the DataFrame. On unsorted data the window picks up random rows and produces meaningless averages. Always sort by the relevant axis (time, score, sequence number) before calling <code>rolling()</code>. The NaN values in the first <code>n-1</code> rows are correct: there isn't enough history to fill those windows yet.
</div>

In time-series work, the pattern is almost always: sort by date, group by entity, then compute a rolling stat per group. The `groupby().transform()` tip from Section 4 applies here too:

In [ ]:
# Per-program 10-student rolling mean of average_marks
# transform keeps the result aligned to the original index
df["marks_prog_rolling10"] = (
    df.sort_values("average_marks")
    .groupby("program")["average_marks"]
    .transform(lambda s: s.rolling(10, min_periods=3).mean())
)
df[["program", "average_marks", "marks_prog_rolling10"]].dropna().head(6)

### 5b. Expressive Filtering with `df.query()`

Boolean masks work well for one or two conditions. For complex multi-condition filters, `df.query()` expresses the same logic as a readable string, closer to how you would say it out loud.

Reference an external variable inside a query string with the `@` prefix:

In [ ]:
# Boolean mask version -- correct but harder to read
mask = (df["midterm_score"] > 70) & (df["final_score"] > 70) & (df["program"].isin(["Engineering", "Sciences"]))
print(f"mask result  : {mask.sum()} rows")

# Same filter as a query string -- reads like a sentence
threshold = 70
result = df.query("midterm_score > @threshold and final_score > @threshold and program in ['Engineering', 'Sciences']")
print(f"query result : {len(result)} rows")

<div style='background:#F5F3FF;border-left:5px solid #7C3AED;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#5B21B6;font-weight:bold'><i class="bi bi-lightbulb-fill"></i> Pro Tip: Use <code>query()</code> for readability, masks for dynamic conditions</span><br><br>
<code>df.query()</code> is evaluated with <code>numexpr</code> when the library is installed, which makes it faster than the equivalent boolean mask on large DataFrames. The downside: the query string is harder to build programmatically. Use <code>query()</code> when you're writing a fixed filter that a reader should understand at a glance; use a mask when the conditions are assembled at runtime from user input or a config.
</div>

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 5 - Smoothed Grade Distribution</span><br><br>

<b>Goal:</b> Sort the full DataFrame by <code>average_marks</code> ascending. Compute a 20-student rolling mean of <code>average_marks</code>. Then use <code>df.query()</code> to find students with a rolling mean above 75 who are in the <code>"Sciences"</code> program. Print how many rows match.
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>df_q = df.sort_values("average_marks").reset_index(drop=True)
df_q["rolling_mean"] = df_q["average_marks"].rolling(20).mean()
result = df_q.query("rolling_mean > 75 and program == 'Sciences'")
print(len(result))</pre>
</div>

In [ ]:
# TODO: rolling mean, then query-filter to Sciences students above rolling threshold
...

## Capstone: Risk and Performance Report

Combine everything from this notebook into one short report: a derived grade column, a categorical breakdown, a text-column check, and a summary statistic, the same operations used in any first pass over a new dataset.

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Capstone Exercise - Risk and Performance Report</span><br><br>

<b>Goal:</b>
<ol>
<li>Confirm every <code>student_id</code> matches the pattern <code>S</code> followed by 4 digits, using <code>.str.match()</code> (Sec. 3)</li>
<li>Print the <code>final_grade</code> distribution as proportions with <code>.value_counts(normalize=True)</code> (Sec. 1, Sec. 2)</li>
<li>Use <code>.agg()</code> to compute the mean <code>average_marks</code> for passed vs failed students separately (Sec. 4)</li>
</ol>
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>valid_ids = df["student_id"].str.match(r"^S\d{4}$").all()

grade_distribution = df["final_grade"].value_counts(normalize=True)

passing = df[df["passed"] == True]["average_marks"].agg(["mean"])  # noqa: E712
failing = df[df["passed"] == False]["average_marks"].agg(["mean"])  # noqa: E712</pre>
</div>

In [ ]:
# TODO: build the risk and performance report described above
...

## Further Reading

| Resource | Why it matters |
|---|---|
| McKinney, W. (2022). *Python for Data Analysis*, 3rd ed. O'Reilly. | Chapter 7 (data cleaning) and Chapter 10 (aggregation) are the canonical references for the operations in this notebook |
| [pandas documentation: Indexing and selecting data](https://pandas.pydata.org/docs/user_guide/indexing.html) | Covers `.loc`, `.iloc`, boolean indexing, and `MultiIndex` in exhaustive detail |
| [pandas documentation: Group by: split-apply-combine](https://pandas.pydata.org/docs/user_guide/groupby.html) | Official guide to `groupby`, `transform`, and `apply`; the examples use the same column types as this notebook |
| Wickham, H. (2014). [Tidy data](https://doi.org/10.18637/jss.v059.i10). *Journal of Statistical Software* 59(10). | The conceptual framework behind `.melt()` and `.pivot_table()` introduced later in Part 10 |


## Summary

| Concept | Key rule |
|---|---|
| `str` dtype | Pandas 3's default for text columns, backed by PyArrow, replacing `object` |
| Copy-on-Write | Selecting a subset always behaves as an independent copy; the original is never modified through it |
| `Series.map()` | One-to-one substitution with a dict or simple function |
| `Series.apply()` | Run any function, including multi-branch logic, one value at a time |
| `DataFrame.apply(axis=1)` | Run a function once per row, with the whole row available |
| Vectorized ops vs `apply` | Prefer a boolean mask or arithmetic when one exists; `apply` is the fallback, not the default |
| `.value_counts(normalize=True)` | Category proportions in one call |
| `.isin()` | Filter rows whose value is in a given list |
| `.astype("category")` | Compact storage for a column with a small, fixed set of values |
| `pd.get_dummies()` | One-hot encode a categorical column before fitting a model |
| `.str` accessor | Required for any string method on a Series; calling the method directly raises `AttributeError` |
| `.str.extract()` | Pull a regex capture group into a new column |
| `.agg([...])` | Compute several statistics for several columns in one call, instead of chaining single-statistic calls |
| `rolling(n)` | Compute a statistic over a sliding window of n rows; sort first so the window is meaningful |
| `expanding()` | Cumulative statistic from the start of the series to the current row |
| `groupby().transform(lambda s: s.rolling(...))` | Per-group rolling stat, result aligned to original index |
| `df.query("expr")` | Readable multi-condition filter; use `@var` to reference external variables |

**Next:** `10-combining-reshaping.ipynb`, covering concatenation, merging, `groupby`, pivot tables, and time series.